# Toy Data — Step 1: Single-Pixel Example (Table 1)

This notebook generates the two toy rasters that reproduce the **single-pixel example** described in:

> Pontius Jr., R.G. and Connors, J., 2009. **Range of categorical associations for comparison of maps with mixed pixels.** *Photogrammetric Engineering & Remote Sensing*, 75(8), pp.963-969. <https://www.ingentaconnect.com/content/asprs/pers/2009/00000075/00000008/art00003>

The example corresponds to Table 1 and Figures 3-5 of the paper.

Each pixel in this example is **mixed**: instead of belonging entirely to one class, it carries a **fractional membership** (a proportion between 0 and 1) to several classes at once. This is the data structure the paper's method is built around, and this notebook stores it explicitly, one band per class, so the underlying equations can be verified directly.

**Data from the paper:**

| Pixel, 1971 | Forest | Agriculture | Wetland | Other |
|---|---|---|---|---|
| Membership | 0.7 | 0.1 | 0.1 | 0.1 |

| Pixel, 1999 | Forest | Built | Remainder |
|---|---|---|---|
| Membership | 0.1 | 0.8 | 0.1 |

Each map is saved as a **multiband** GeoTIFF, 1x1 pixel, `dtype=float32`, where each band corresponds to one class (band order follows the order of the membership dictionary).

In [ ]:
# -----------------------------------------------------------------------------
# 01. INSTALL REQUIRED LIBRARIES
# -----------------------------------------------------------------------------
!pip install rasterio -q

# -----------------------------------------------------------------------------
# 02. IMPORT LIBRARIES AND MOUNT GOOGLE DRIVE
# -----------------------------------------------------------------------------
import os
import numpy as np
import pandas as pd
import rasterio
from rasterio.transform import from_origin
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')


In [ ]:
# -----------------------------------------------------------------------------
# 03. DEFINE MULTIBAND MEMBERSHIP RASTER FUNCTION
# -----------------------------------------------------------------------------
def create_membership_raster(
    class_memberships: dict,
    output_path: str,
    pixel_size: float = 10.0,
    crs: str = "EPSG:4326"
) -> str:
    """
    Writes a single-pixel, multiband GeoTIFF where each band stores the
    fractional membership (0-1) of that pixel to one category. This is the
    data structure required by Pontius & Connors (2009) for mixed /
    soft-classified pixels: for each pixel, the quantity of each category is
    known (the membership value), but the sub-pixel spatial allocation is not.

    Parameters
    ----------
    class_memberships : dict
        Ordered mapping of {class_name: membership_value}. Values should sum
        to 1.0 for a single pixel (as in the paper's Table 1 example).
    output_path : str
        Full file path (including filename) for the output GeoTIFF.
    pixel_size : float, optional
        Spatial resolution of the single pixel (default 10.0).
    crs : str, optional
        Coordinate Reference System string (default 'EPSG:4326').

    Returns
    -------
    str
        The output file path.
    """
    # 03.1 Ensure output directory exists
    os.makedirs(
        os.path.dirname(output_path),
        exist_ok=True
    )

    # 03.2 Sanity check: memberships should sum to 1 (as in the paper)
    total_membership = sum(class_memberships.values())
    if not np.isclose(total_membership, 1.0):
        print(
            f"[WARNING] Class memberships sum to {total_membership:.3f}, "
            "not 1.0. The paper assumes membership per pixel sums to 1."
        )

    # 03.3 Define spatial transform (arbitrary 1x1 pixel origin at 0, 10)
    transform = from_origin(
        0.0,
        pixel_size,
        pixel_size,
        pixel_size
    )

    class_names = list(class_memberships.keys())
    num_bands = len(class_names)

    # 03.4 Define GeoTIFF metadata profile (float32, one band per class)
    meta = {
        "driver": "GTiff",
        "height": 1,
        "width": 1,
        "count": num_bands,
        "dtype": "float32",
        "crs": crs,
        "transform": transform,
        "nodata": -1.0
    }

    # 03.5 Write raster file to disk, one band per class, band description = class name
    with rasterio.open(output_path, "w", **meta) as dst:
        for band_idx, class_name in enumerate(class_names, start=1):
            band_array = np.array(
                [[class_memberships[class_name]]],
                dtype=np.float32
            )
            dst.write(band_array, band_idx)
            dst.set_band_description(band_idx, class_name)

    print(f"[LOG] Exported: {output_path} (bands: {class_names})")
    return output_path


In [ ]:
# -----------------------------------------------------------------------------
# 04. DATA INITIALIZATION (Table 1 / Figures 3-5 of Pontius & Connors, 2009)
# -----------------------------------------------------------------------------

# 04.1 1971 pixel membership: Forest, Agriculture, Wetland, Other
membership_1971 = {
    "forest": 0.7,
    "agriculture": 0.1,
    "wetland": 0.1,
    "other": 0.1
}

# 04.2 1999 pixel membership: Forest, Built, Remainder
membership_1999 = {
    "forest": 0.1,
    "built": 0.8,
    "remainder": 0.1
}

# 04.3 Set output folder destination inside Google Drive
drive_output_folder = "/content/drive/MyDrive/toyData/step1_singlePixel"

# 04.4 Generate the two single-pixel membership rasters
path_1971 = create_membership_raster(
    class_memberships=membership_1971,
    output_path=os.path.join(drive_output_folder, "map_1971.tif"),
    pixel_size=10.0,
    crs="EPSG:4326"
)

path_1999 = create_membership_raster(
    class_memberships=membership_1999,
    output_path=os.path.join(drive_output_folder, "map_1999.tif"),
    pixel_size=10.0,
    crs="EPSG:4326"
)

print("\n[SUCCESS] Both single-pixel membership rasters have been generated and saved to Google Drive!")


## Verification — Equations 1, 3 and 5

Before moving on to Step 2, it is worth confirming that the data matches **Table 1** of the paper, by re-implementing the three equations directly for a single pixel:

- **Greatest** (Eq. 1): $M_{ij} = \min(X_i, Y_j)$
- **Random** (Eq. 3): $R_{ij} = X_i \cdot Y_j$
- **Least** (Eq. 5): $L_{ij} = \max(0, X_i + Y_j - 1)$

where $X_i$ is the membership of class $i$ in the 1971 map and $Y_j$ is the membership of class $j$ in the 1999 map.

In [ ]:
# -----------------------------------------------------------------------------
# 05. VERIFY AGAINST TABLE 1 OF THE PAPER
# -----------------------------------------------------------------------------
def read_membership_raster(path: str) -> dict:
    """Reads a multiband single-pixel membership raster back into a dict."""
    with rasterio.open(path) as src:
        return {
            src.descriptions[b]: float(src.read(b + 1)[0, 0])
            for b in range(src.count)
        }

def greatest_matrix(X: dict, Y: dict) -> pd.DataFrame:
    return pd.DataFrame(
        {yj: [min(X[xi], Y[yj]) for xi in X] for yj in Y},
        index=list(X.keys())
    )

def random_matrix(X: dict, Y: dict) -> pd.DataFrame:
    return pd.DataFrame(
        {yj: [X[xi] * Y[yj] for xi in X] for yj in Y},
        index=list(X.keys())
    )

def least_matrix(X: dict, Y: dict) -> pd.DataFrame:
    return pd.DataFrame(
        {yj: [max(0.0, X[xi] + Y[yj] - 1.0) for xi in X] for yj in Y},
        index=list(X.keys())
    )

# 05.1 Read the two rasters back (round-trip check)
X = read_membership_raster(path_1971)
Y = read_membership_raster(path_1999)

print("1971 membership (read from raster):", X)
print("1999 membership (read from raster):", Y)

# 05.2 Build the three matrices
M = greatest_matrix(X, Y)
R = random_matrix(X, Y)
L = least_matrix(X, Y)

print("\nGreatest matrix (should match the top row of each cell in Table 1):")
display(M)

print("\nRandom matrix (should match the middle row of each cell in Table 1):")
display(R)

print("\nLeast matrix (should match the bottom row of each cell in Table 1):")
display(L)

# 05.3 Expected values transcribed from Table 1 of Pontius & Connors (2009)
expected_M = pd.DataFrame(
    {"forest": [0.10, 0.10, 0.10, 0.10], "built": [0.70, 0.10, 0.10, 0.10], "remainder": [0.10, 0.10, 0.10, 0.10]},
    index=["forest", "agriculture", "wetland", "other"]
)
expected_R = pd.DataFrame(
    {"forest": [0.07, 0.01, 0.01, 0.01], "built": [0.56, 0.08, 0.08, 0.08], "remainder": [0.07, 0.01, 0.01, 0.01]},
    index=["forest", "agriculture", "wetland", "other"]
)
expected_L = pd.DataFrame(
    {"forest": [0.00, 0.00, 0.00, 0.00], "built": [0.50, 0.00, 0.00, 0.00], "remainder": [0.00, 0.00, 0.00, 0.00]},
    index=["forest", "agriculture", "wetland", "other"]
)

print("\n[CHECK] Greatest matches Table 1:", np.allclose(M.values, expected_M.values))
print("[CHECK] Random matches Table 1:", np.allclose(R.values, expected_R.values))
print("[CHECK] Least matches Table 1:", np.allclose(L.values, expected_L.values))
